In [58]:
import os
import json
import re
from html import unescape

# Define your folder path
folder_path = "C:/Users/HP/Desktop/NLP-PROJECT/data" # FOLDER JMA3A not file 


In [59]:
def list_article_attributes(data):
    """
    Returns a set of all unique attributes present in the articles.
    """
    attributes = set()

    if "articles" in data:
        for article in data["articles"]:
            attributes.update(article.keys())

    return attributes


In [ ]:
import re
from html import unescape

def get_articles_list(data):
    """
    Extract articles list from data whether it's a list or has 'articles' key
    """
    if isinstance(data, list):
        return data
    elif isinstance(data, dict) and "articles" in data:
        return data["articles"]
    else:
        return [data]

def set_articles_list(original_data, articles):
    """
    Set articles list back into the original structure
    """
    if isinstance(original_data, list):
        return articles
    elif isinstance(original_data, dict) and "articles" in original_data:
        original_data["articles"] = articles
        return original_data
    else:
        return articles[0] if articles else None

def remove_retracted_articles(data):
    """
    Remove articles from data if 'is_retracted' is True.
    """
    articles = get_articles_list(data)
    
    # Keep only articles where is_retracted is False or missing
    filtered_articles = [
        article for article in articles
        if not article.get("is_retracted", False)
    ]
    
    return set_articles_list(data, filtered_articles)

def remove_unwanted_attributes(data):
    """
    Remove specified attributes from each article in the JSON data.
    """
    unwanted_attrs = [
        'id', 'doi', 'ids', 'publication_year', 'institution_assertions',
        'corresponding_author_ids', 'is_corresponding', 'corresponding_institution_ids',
        'apc_list', 'apc_paid', 'has_fulltext', 'biblio', 'fulltext_origin',
        'is_paratext', 'concepts', 'locations_count', 'locations', 'best_oa_location',
        'sustainable_development_goals', 'grants', 'datasets', 'versions',
        'referenced_works_count', 'referenced_works', 'related_works',
        'cited_by_api_url', 'counts_by_year', 'updated_date', 'created_date',
        'type', 'type_crossref', 'indexed_in', 'open_access', 'mesh'
    ]

    articles = get_articles_list(data)
    
    for article in articles:
        for attr in unwanted_attrs:
            if attr in article:
                del article[attr]

    return set_articles_list(data, articles)

def handle_missing_fields(data, placeholder="N/A"):
    """
    Check important text fields in each article.
    If a field is missing or empty, fill it with a placeholder.
    """
    fields_to_check = ["title", "abstract"]

    articles = get_articles_list(data)
    
    for article in articles:
        for field in fields_to_check:
            if field not in article or not article[field] or article[field].strip() == "":
                article[field] = placeholder

    return set_articles_list(data, articles)

def normalize_text(data):
    """
    Normalize text fields ('title' and 'abstract') in each article:
    - Lowercase
    - Remove extra spaces and line breaks
    - Remove HTML tags and special characters
    """
    def clean_text(text):
        if not text:
            return ""
        # Decode HTML entities
        text = unescape(text)
        # Remove HTML tags
        text = re.sub(r"<.*?>", " ", text)
        # Remove non-printable or weird characters
        text = re.sub(r"[^ -~\n]", " ", text)
        # Replace multiple spaces or newlines with a single space
        text = re.sub(r"\s+", " ", text)
        # Strip leading/trailing spaces
        return text.strip().lower()

    articles = get_articles_list(data)
    
    for article in articles:
        if "title" in article:
            article["title"] = clean_text(article["title"])
        if "abstract" in article:
            article["abstract"] = clean_text(article["abstract"])

    return set_articles_list(data, articles)

def handle_abstract(data):
    """
    Reconstruct the abstract from 'abstract_inverted_index' for each article.
    Replaces 'abstract_inverted_index' with 'abstract'.
    """
    articles = get_articles_list(data)
    
    for article in articles:
        inverted_index = article.get("abstract_inverted_index")

        if inverted_index:
            # Create a dict with position as key and word as value
            position_word = {}
            for word, positions in inverted_index.items():
                for pos in positions:
                    position_word[pos] = word

            # Reconstruct abstract based on sorted positions
            abstract_words = [position_word[pos] for pos in sorted(position_word.keys())]
            article["abstract"] = " ".join(abstract_words)

        # Remove the original inverted index in any case
        if "abstract_inverted_index" in article:
            del article["abstract_inverted_index"]

    return set_articles_list(data, articles)

def handle_primary_location(data):
    """
    Simplify the 'primary_location' field into 'publication_info'.
    """
    articles = get_articles_list(data)
    
    for article in articles:
        primary = article.get("primary_location")
        if primary is not None:
            source = primary.get("source")
            if source is not None:
                article["publication_info"] = {
                    "host_organization_name": source.get("host_organization_name"),
                    "host_organization_lineage_names": source.get("host_organization_lineage_names"),
                    "type": source.get("type")
                }
            else:
                article["publication_info"] = {
                    "host_organization_name": None,
                    "host_organization_lineage_names": None,
                    "type": None
                }
            del article["primary_location"]
    
    return set_articles_list(data, articles)

def handle_authorships(data):
    """
    Simplify the 'authorships' field for each article.
    Keeps only: author name, author position, institutions, countries, and is_corresponding.
    """
    articles = get_articles_list(data)
    
    for article in articles:
        if "authorships" in article:
            simplified_authors = []
            for auth in article["authorships"]:
                author_name = auth.get("author", {}).get("display_name")
                author_position = auth.get("author_position")
                institutions = [
                    inst.get("display_name") if isinstance(inst, dict) else inst
                    for inst in auth.get("institutions", [])
                ]
                countries = auth.get("countries", [])
                is_corresponding = auth.get("is_corresponding", False)

                simplified_authors.append({
                    "name": author_name,
                    "position": author_position,
                    "institutions": institutions,
                    "countries": countries,
                    "is_corresponding": is_corresponding
                })

            article["authorships"] = simplified_authors

    return set_articles_list(data, articles)

def handle_topics(data, max_topics=3):
    """
    Simplify primary_topic and topics for each article.
    Keeps only human-readable display names.
    """
    articles = get_articles_list(data)
    
    for article in articles:
        # Handle primary_topic
        primary = article.get("primary_topic")
        if primary:
            article["primary_topic_name"] = primary.get("display_name")
            del article["primary_topic"]

        # Handle topics list
        topics_list = article.get("topics", [])
        if topics_list:
            article["topics_names"] = [t.get("display_name") for t in topics_list[:max_topics]]
            del article["topics"]

    return set_articles_list(data, articles)

def handle_keywords(data, max_keywords=10):
    """
    Simplify 'keywords' for each article.
    Keeps only the top N keyword display names.
    """
    articles = get_articles_list(data)
    
    for article in articles:
        keywords_list = article.get("keywords", [])
        if keywords_list:
            article["keywords_names"] = [k.get("display_name") for k in keywords_list[:max_keywords]]
            del article["keywords"]

    return set_articles_list(data, articles)

def build_classification_path(article):
    """
    Build classification path from domain -> field -> subfield -> topic.
    """
    classification_path = []

    # Try both possible field names since handle_topics renames primary_topic
    if 'primary_topic' in article and article['primary_topic']:
        primary_topic = article['primary_topic']
    elif 'primary_topic_name' in article and article['primary_topic_name']:
        # If handle_topics already ran, we only have the name
        return [article['primary_topic_name']]  # Simplified path
    else:
        return []

    if primary_topic:
        classification_path.extend([
            primary_topic['domain']['display_name'],
            primary_topic['field']['display_name'],
            primary_topic['subfield']['display_name'],
            primary_topic['display_name']
        ])

    return classification_path

def handle_classification_path(data):
    """
    Add a 'classification_path' field to each article using build_classification_path().
    """
    articles = get_articles_list(data)
    
    for article in articles:
        article["classification_path"] = build_classification_path(article)
    
    return set_articles_list(data, articles)

def remove_incomplete_articles(data):
    """
    Remove articles with missing or empty:
    - title
    - abstract
    - authorships (must have at least one)
    Also removes articles where these fields are completely absent.
    """
    articles = get_articles_list(data)
    
    cleaned_articles = []
    for article in articles:
        title = article.get("title")
        abstract = article.get("abstract")
        authorships = article.get("authorships", [])
        citations = article.get("cited_by_count", None)

        # Check if title is missing or empty
        if not title or not str(title).strip():
            continue
        # Check if abstract is missing or empty
        if not abstract or not str(abstract).strip():
            continue
        # Check if authorships is missing or empty
        if not isinstance(authorships, list) or len(authorships) == 0:
            continue
        # Only remove if citations is completely missing (None), not if it's 0
        if citations is None:
            continue

        cleaned_articles.append(article)

    print(f" Kept {len(cleaned_articles)} / {len(articles)} valid articles.")
    return set_articles_list(data, cleaned_articles)

In [ ]:
import json
import os

# Add more functions as needed
preprocessing_functions = [
    remove_retracted_articles,
    remove_unwanted_attributes,
    handle_abstract,
    handle_primary_location,
    handle_classification_path,
    handle_authorships,
    handle_topics,
    handle_keywords,
    handle_missing_fields,
    normalize_text,
    remove_incomplete_articles
]

# Define your folder path
folder_path = "C:/Users/HP/Desktop/NLP-PROJECT/data"

# Loop through files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        file_path = os.path.join(folder_path, filename)
        print(f"Processing {filename}...")

        try:
            # Read JSON data
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Apply preprocessing functions
            for func in preprocessing_functions:
                data = func(data)

            # Save the processed data back to JSON
            with open(file_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=4)

            print(f" Successfully processed {filename}")

        except Exception as e:
            print(f" Error processing {filename}: {str(e)}")
            continue

print("🎉 All files processed successfully!")

Processing cc.json...
✅ Kept 4981 / 4997 valid articles.
✅ Successfully processed cc.json
🎉 All files processed successfully!
